# Notebook Summary

This notebook links an **Azure Databricks Workspace (Unity Catalog)** to the current Fabric **Lakehouse** by creating external Delta tables in the Lakehouse metadata layer that point to the Unity Catalog table storage locations.

**High-level purpose**  
Automatically mirror Unity Catalog schemas and tables into the Lakehouse so that Databricks Unity Catalog tables can be queried from Fabric as if they were native Lakehouse tables.

---

## Main steps

1. **Parameter setup**  
   - `notebook_connection_id`: ID of the notebook-level Fabric connection of type **Azure Databricks Workspace**.  
   - `unity_catalog_name_databricks`: Name of the Unity Catalog catalog whose tables should be linked.  
   - `drop_table_if_exists`: If `True`, existing Lakehouse tables with the same names are dropped before they are recreated.

2. **Environment setup**  
   - Installs the Python package `databricks-sdk`.  
   - Imports required libraries: `json`, `time`, `requests`, `sempy.fabric`, and `WorkspaceClient` from `databricks.sdk`.

3. **Read notebook connection details**  
   - Uses `notebookutils.connections.getCredential` to read the notebook connection.  
   - Extracts the OAuth access token (`AccessToken`) used to authenticate against Databricks.  
   - Retrieves the underlying Fabric connection ID (`datasourceId`).

4. **Query Fabric Connection metadata**  
   - Uses `notebookutils.credentials.getToken("pbi")` to obtain a Fabric API token.  
   - Calls the Fabric Connections REST API (`/v1/connections/{id}`) to get connection details.  
   - Reads the Databricks workspace hostname from `connectionDetails.path`.

5. **Initialize Databricks Workspace Client**  
   - Creates a `WorkspaceClient` with the Databricks host and OAuth token (passed as `auth_type="pat"`, even though it is an OAuth token).  
   - Verifies access by calling `w.current_user.me()` and printing the user name.

6. **Resolve Unity Catalog catalog**  
   - Retrieves the catalog object via `w.catalogs.get(unity_catalog_name_databricks)`.  
   - Prints `catalog.full_name` for reference.

7. **Load schemas and tables from Unity Catalog**  
   - Lists all schemas in the catalog with `w.schemas.list(...)`, excluding `information_schema`.  
   - For each schema, lists all tables via `w.tables.list(catalog_name=catalog.name, schema_name=...)`.  
   - Builds a `schemas` dictionary (schema name → schema id) and a list `all_tables` with all table objects.

8. **Create schemas in the Lakehouse**  
   - For each schema name, runs a Spark SQL command:

     ```sql
     CREATE SCHEMA IF NOT EXISTS <schema_name>;
     ```

   - Ensures that corresponding schemas exist in the Lakehouse before tables are created.

9. **Drop existing tables in the Lakehouse (optional)**  
   - If `drop_table_if_exists` is `True`, iterates over `all_tables` and executes:

     ```sql
     DROP TABLE IF EXISTS <schema_name>.<table_name>
     ```

   - Waits briefly (`time.sleep(10)`) to allow metadata to be updated after the drop operation.

10. **Create external Delta tables in the Lakehouse**  
    - For each Unity Catalog table:
      - Builds the fully qualified Unity Catalog name: `catalog_name.schema_name.table_name`.  
      - Calls `w.tables.get(full_name=...)` to retrieve table details.  
      - Reads the underlying storage location from `table_detail.storage_location`.  
      - Constructs the Lakehouse table identifier: `` `schema_name`.`table_name` ``.  
      - Optionally drops the Lakehouse table again (safety step if `drop_table_if_exists` is `True`).  
      - Executes a Spark SQL statement to create an **external Delta table** pointing to the Unity Catalog storage path:

        ```sql
        CREATE EXTERNAL TABLE `schema_name`.`table_name`
        USING DELTA
        LOCATION '<table_path>'
        ```

---

## Result

After running the notebook:
- All Unity Catalog schemas (except `information_schema`) are created in the Lakehouse.  
- For each Unity Catalog table, a matching **external Delta table** is created in the Lakehouse that directly references the Unity Catalog Delta files.  
- You can now query those Unity Catalog tables from Fabric (Spark/Lakehouse) as if they were native Lakehouse tables, without duplicating the underlying data.


### Parameters

Set the main parameters for this notebook (Databricks connection, Unity Catalog catalog name, and whether to drop existing Lakehouse tables before recreating them).

In [1]:
notebook_connection_id = "b8e512e9-9b74-488e-b5ce-9a8a82fe372e" 
unity_catalog_name_databricks = "azure"
drop_table_if_exists = True

StatementMeta(, 2a0c5b54-942e-40fc-8e7c-51e476d37502, 3, Finished, Available, Finished, False)

### Install Databricks SDK

Install the `databricks-sdk` Python package so the notebook can call the Databricks REST APIs via the official SDK.
- Link to Databricks SDK [Databricks SDK](https://databricks-sdk-py.readthedocs.io/en/stable/getting-started.html)

In [ ]:
!pip install databricks-sdk

### Import libraries

Import all required Python libraries (JSON handling, time, HTTP requests, Fabric helper `sempy.fabric`, and the `WorkspaceClient` from the Databricks SDK).

In [3]:
import json
import time
import requests

import sempy.fabric as fabric
from databricks.sdk import WorkspaceClient

StatementMeta(, 2a0c5b54-942e-40fc-8e7c-51e476d37502, 5, Finished, Available, Finished, False)

### Read notebook connection

Read the notebook-scoped Fabric connection, extract the Databricks OAuth access token, and get the underlying Fabric connection ID.

In [4]:
connection_credential = notebookutils.connections.getCredential(notebook_connection_id)
credential_dict = json.loads(connection_credential['credential'])

databricks_token = [x["value"] for x in credential_dict["credentialData"] if x["name"] == "AccessToken"][0]
connection_id_fabric = connection_credential["datasourceId"] # actual Fabric connection ID

StatementMeta(, 2a0c5b54-942e-40fc-8e7c-51e476d37502, 6, Finished, Available, Finished, False)

### Read Fabric connection details

Call the Fabric Connections REST API to retrieve full metadata for the selected connection and read the Databricks workspace hostname from it.

In [5]:
token = notebookutils.credentials.getToken("pbi")

headers = {
    "Authorization": f"Bearer {token}"
}

resp = requests.get(f"https://api.fabric.microsoft.com/v1/connections/{connection_id_fabric}", headers = headers)

conn_details = resp.json()

hostname = conn_details["connectionDetails"]["path"]

StatementMeta(, 2a0c5b54-942e-40fc-8e7c-51e476d37502, 7, Finished, Available, Finished, False)

### Setup Databricks WorkspaceClient

Create a Databricks `WorkspaceClient` using the hostname and OAuth token, then verify that the connection works by printing the current user.

In [ ]:
w = WorkspaceClient(
    host = hostname,
    token = databricks_token,
    auth_type = "pat", 
)

me = w.current_user.me()
print(me.user_name)

### Get Unity Catalog catalog

Retrieve the Unity Catalog catalog object identified by `unity_catalog_name_databricks` and print its full name for verification.

In [ ]:
catalog = w.catalogs.get(unity_catalog_name_databricks)
print(f"{catalog.full_name = }")

### Load schemas and tables from Unity Catalog

List all schemas (except `information_schema`) and tables from the selected Unity Catalog catalog and build helper collections: a schema dictionary and a flat list of all tables.

In [ ]:
uc_schemas = [s for s in w.schemas.list(catalog_name=catalog.name) if s.name != "information_schema"]

schemas = {}
all_tables = []
for uc_schema in uc_schemas:
    schemas[uc_schema.name] = uc_schema.schema_id
    tables = w.tables.list(catalog_name=catalog.name, schema_name=uc_schema.name)
    
    all_tables.extend([t for t in tables])

[(t.schema_name, t.name, t.table_id) for t in all_tables]

### Create schemas in Lakehouse

Ensure that each Unity Catalog schema also exists in the Lakehouse by running `CREATE SCHEMA IF NOT EXISTS` for every schema found.

In [ ]:
for schema in schemas:
    sql_cmd = f"CREATE SCHEMA IF NOT EXISTS {schema};"
    print(sql_cmd)
    spark.sql(sql_cmd)

### Drop existing Lakehouse tables (optional)

Optionally (controlled by `drop_table_if_exists`) drop all existing Lakehouse tables that correspond to the Unity Catalog tables, then wait briefly so metadata can refresh.

In [ ]:
for table in all_tables:
    table_name = f"{table.schema_name}.{table.name}"
    
    if drop_table_if_exists:
        sql_cmd = f"DROP TABLE IF EXISTS {table_name}"
        print(sql_cmd)
        spark.sql(sql_cmd)

time.sleep(10)

### Create external Delta tables in Lakehouse

For each Unity Catalog table, read its storage location and create a matching external Delta table in the Lakehouse that points to that path.

In [ ]:
for table in all_tables:

    full_name = (
        f"{table.catalog_name}."
        f"{table.schema_name}."
        f"{table.name}"
    )

    table_detail = w.tables.get(full_name=full_name)

    table_path = table_detail.storage_location

    table_name = f"`{table.schema_name}`.`{table.name}`"

    sql_cmd = f"""
CREATE EXTERNAL TABLE {table_name}
USING DELTA
LOCATION '{table_path}'
"""

    print(sql_cmd)
    spark.sql(sql_cmd)
